In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\Historical Image Restoration


In [2]:
import torch

from src.models import RestorationUNet

print("PyTorch:", torch.__version__)


PyTorch: 2.14.0+cpu


In [3]:
model = RestorationUNet(
    in_channels=3,
    out_channels=3,
    base_channels=64,
    num_levels=4,
    residual_blocks_per_level=2,
)

print(model)


RestorationUNet(
  (encoder1): EncoderBlock(
    (conv): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
    )
    (residual): Sequential(
      (0): ResidualBlock(
        (block): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): ReLU(inplace=True)
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
      )
      (1): ResidualBlock(
        (block): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): ReLU(inplace=True)
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
      )
    )
    (downsample): Conv2d(64, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  )
  (encoder2): EncoderBlock(
    (conv): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)

In [4]:
total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 54,854,275


In [5]:
dummy_input = torch.rand(2, 3, 256, 256)

with torch.no_grad():
    output = model(dummy_input)

print("Input shape :", dummy_input.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([2, 3, 256, 256])
Output shape: torch.Size([2, 3, 256, 256])


In [6]:
print("Output min:", output.min().item())
print("Output max:", output.max().item())

Output min: 0.4654364585876465
Output max: 0.5347477793693542


In [7]:
from src.models import L1ReconstructionLoss

criterion = L1ReconstructionLoss()

target = torch.rand_like(output)

loss = criterion(output, target)

print("L1 loss:", loss.item())

L1 loss: 0.2500125765800476


In [9]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

model.train()

input_image = torch.rand(2, 3, 256, 256)
target_image = torch.rand(2, 3, 256, 256)

optimizer.zero_grad()

prediction = model(input_image)

loss = criterion(prediction, target_image)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("Backward pass: SUCCESS")
print("Optimizer step: SUCCESS")

Loss: 0.25033584237098694
Backward pass: SUCCESS
Optimizer step: SUCCESS


In [10]:
import platform
import psutil

print("CPU:", platform.processor())
print("Logical CPU cores:", psutil.cpu_count(logical=True))
print("Physical CPU cores:", psutil.cpu_count(logical=False))
print("RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))

CPU: Intel64 Family 6 Model 142 Stepping 12, GenuineIntel
Logical CPU cores: 8
Physical CPU cores: 4
RAM (GB): 7.88


In [11]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

CUDA available: False
CUDA device count: 0


In [12]:
import time
import torch

model = model.cpu()
dummy_input = torch.rand(1, 3, 256, 256)

start = time.time()

with torch.no_grad():
    output = model(dummy_input)

elapsed = time.time() - start

print("Output shape:", output.shape)
print(f"Single-image CPU inference time: {elapsed:.2f} seconds")

Output shape: torch.Size([1, 3, 256, 256])
Single-image CPU inference time: 2.17 seconds


In [13]:
from src.models import RestorationUNet

model = RestorationUNet(
    in_channels=3,
    out_channels=3,
    base_channels=32,
    num_levels=4,
    residual_blocks_per_level=2,
)

In [14]:
total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 13,717,827


In [17]:
dummy_input = torch.rand(1, 3, 256, 256)

start = time.time()

with torch.no_grad():
    output = model(dummy_input)

elapsed = time.time() - start

print("Input shape :", dummy_input.shape)
print("Output shape:", output.shape)
print(f"Single-image CPU inference time: {elapsed:.2f} seconds")

Input shape : torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Single-image CPU inference time: 0.55 seconds


In [20]:
from datasets import load_dataset

dataset = load_dataset("joshuachin/openphoto-restore-dataset")

In [21]:
from torch.utils.data import DataLoader
from src.datasets.openphoto_dataset import OpenPhotoRestoreDataset

train_dataset = OpenPhotoRestoreDataset(
    dataset["train"],
    crop_size=256,
    training=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
)

In [22]:
model.train()

batch = next(iter(train_loader))

damaged = batch["damaged"]
clean = batch["clean"]

prediction = model(damaged)

print("Damaged:", damaged.shape)
print("Prediction:", prediction.shape)
print("Clean:", clean.shape)

Damaged: torch.Size([4, 3, 256, 256])
Prediction: torch.Size([4, 3, 256, 256])
Clean: torch.Size([4, 3, 256, 256])


In [23]:
from src.models import L1ReconstructionLoss

criterion = L1ReconstructionLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

optimizer.zero_grad()

prediction = model(damaged)

loss = criterion(prediction, clean)

loss.backward()

optimizer.step()

print("Real OpenPhoto batch training step")
print("Loss:", loss.item())
print("Backward: SUCCESS")
print("Optimizer step: SUCCESS")

Real OpenPhoto batch training step
Loss: 0.25962552428245544
Backward: SUCCESS
Optimizer step: SUCCESS
